# NeuroScan AI · train the U-Net only (resumable)

Use this when the VGG16 classifier is already trained. Produces:

| File | What it is |
|---|---|
| `unet_vgg16.pt` | U-Net with a pretrained VGG16 encoder, trained on BraTS 2023 slices |
| `unet_vgg16.metrics.json` | Dice for whole tumor, tumor core, enhancing tumor |

**Runs on Colab or Kaggle.** If Colab says your GPU usage limit is reached, use Kaggle (30 free GPU hours per week):

- **Kaggle:** kaggle.com → Create → New Notebook → File → Import Notebook → upload this file.
  In the right panel: **Accelerator → GPU T4 x2** and **Internet → On** (needs a phone-verified account). Then **Run All**.
  Download `neuroscan_unet.zip` from the **Output** panel (folder `/kaggle/working`) when it finishes.
- **Colab:** Runtime → Change runtime type → T4 GPU → Run all. Allow the Google Drive popup.

**Resumable:** after every epoch a resume file is saved (Google Drive on Colab, `/kaggle/working` on Kaggle).
If the session stops, run all again: training continues from the last finished epoch.

About 20–30 minutes on a T4. Research and education only.

In [ ]:
# Settings for the U-Net run. Works on Colab and Kaggle.
import os, sys, json, time, random, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("NEUROSCAN_SMOKE") == "1"   # tiny run used to test this notebook on a CPU

WORK = Path(os.environ.get("NEUROSCAN_WORK", "/kaggle/working" if IN_KAGGLE else "/content"))
BRATS_RAW = WORK / "brats2023"
OUT = WORK / "neuroscan_weights"

# Save every checkpoint straight to Google Drive, so a dropped Colab session loses nothing.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/NeuroScanAI"

BRATS_CASES = 2 if SMOKE else 80             # of 1,251 available; more = better but slower
UNET_EPOCHS = 2 if SMOKE else 12
SEG_BATCH = 2 if SMOKE else 8              # 8 fits a 16 GB T4 with mixed precision
SEED = 42

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "huggingface_hub"], check=True)
    if IN_COLAB and SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")   # a Google sign-in popup appears: allow access
        OUT = Path(DRIVE_DIR) / "weights"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import VGG16_Weights, vgg16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
OUT.mkdir(parents=True, exist_ok=True)
print("saving to:", OUT)
print("device:", DEVICE, torch.cuda.get_device_name(0) if AMP else "", "| smoke test" if SMOKE else "")
if not AMP and not SMOKE:
    print("WARNING: no GPU. Runtime → Change runtime type → T4 GPU, then run again.")

## Model definitions

Copied from `backend/app/models/` so the saved weights load straight into the backend. Do not rename layers.

In [ ]:
CLASS_NAMES = ("glioma", "meningioma", "no_tumor", "pituitary")
SEG_CLASSES = ("background", "necrotic_core", "edema", "enhancing")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def build_vgg16_classifier(num_classes=len(CLASS_NAMES), pretrained=True):
    model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetVGG16(nn.Module):
    def __init__(self, num_classes=len(SEG_CLASSES), pretrained_encoder=True):
        super().__init__()
        features = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained_encoder else None).features
        self.enc1 = features[0:4]
        self.enc2 = features[4:9]
        self.enc3 = features[9:16]
        self.enc4 = features[16:23]
        self.bottleneck = features[23:30]
        self.up4 = nn.ConvTranspose2d(512, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512 + 512, 256)
        self.up3 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256 + 256, 128)
        self.up2 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128 + 128, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 64, 32)
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        b = self.bottleneck(s4)
        d = self.dec4(torch.cat([self.up4(b), s4], 1))
        d = self.dec3(torch.cat([self.up3(d), s3], 1))
        d = self.dec2(torch.cat([self.up2(d), s2], 1))
        d = self.dec1(torch.cat([self.up1(d), s1], 1))
        return self.head(d)

print("models defined")

## 1 · Segmentation data (BraTS 2023 3D volumes, sliced to 2D)

In [ ]:
from huggingface_hub import list_repo_files, snapshot_download

workers = 2 if (IN_COLAB or IN_KAGGLE) else 0
import nibabel as nib

REPO = "MedOtter/brats2023-gli-dataset"
local_cases = sorted({p.parent.name for p in BRATS_RAW.rglob("*-seg.nii.gz")})
if len(local_cases) < BRATS_CASES:
    files = list_repo_files(REPO, repo_type="dataset")
    cases = sorted({f.split("/")[1] for f in files if f.endswith("-seg.nii.gz")})[:BRATS_CASES]
    t = time.time()
    snapshot_download(REPO, repo_type="dataset", local_dir=BRATS_RAW, max_workers=16,
                      allow_patterns=[f"*/{c}/{c}-{m}.nii.gz" for c in cases for m in ("t1c", "seg")])
    print(f"downloaded {len(cases)} cases in {time.time() - t:.0f}s")

def normalise(vol):
    brain = vol[vol > 0]
    if brain.size == 0:
        return np.zeros_like(vol, dtype=np.uint8)
    lo, hi = np.percentile(brain, (1, 99))
    return (np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)

SIZE = 256

def load_case(seg_path):
    case = seg_path.name.removesuffix("-seg.nii.gz")
    img = normalise(np.asarray(nib.load(str(seg_path.with_name(f"{case}-t1c.nii.gz"))).dataobj, dtype=np.float32))
    seg = np.asarray(nib.load(str(seg_path)).dataobj).astype(np.uint8)
    tumor = [z for z in range(seg.shape[2]) if seg[:, :, z].any()]
    empty = [z for z in range(seg.shape[2]) if img[:, :, z].max() > 10 and not seg[:, :, z].any()]
    keep = tumor + random.Random(case).sample(empty, min(len(empty), len(tumor) // 6))
    xs, ys = [], []
    for z in keep:
        # Same orientation and resizing as the backend (rot90 of the axial slice).
        xs.append(np.asarray(Image.fromarray(np.rot90(img[:, :, z]).copy()).resize((SIZE, SIZE), Image.BILINEAR)))
        ys.append(np.asarray(Image.fromarray(np.rot90(seg[:, :, z]).copy()).resize((SIZE, SIZE), Image.NEAREST)))
    return np.stack(xs), np.stack(ys)

seg_paths = sorted(BRATS_RAW.rglob("*-seg.nii.gz"))[:BRATS_CASES]
random.Random(SEED).shuffle(seg_paths)
n_val = max(1, round(len(seg_paths) * 0.2))
splits = {"val": seg_paths[:n_val], "train": seg_paths[n_val:]}
arrays = {}
for split, paths in splits.items():
    parts = [load_case(p) for p in paths]
    arrays[split] = (np.concatenate([a for a, _ in parts]), np.concatenate([b for _, b in parts]))
    print(f"{split}: {len(paths)} patients, {len(arrays[split][0])} slices")

class SliceSet(Dataset):
    def __init__(self, x, y, augment):
        self.x, self.y, self.augment = x, y, augment
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        img = self.x[i].astype(np.float32) / 255.0
        mask = self.y[i].astype(np.int64)
        if self.augment and random.random() < 0.5:
            img, mask = img[:, ::-1].copy(), mask[:, ::-1].copy()
        rgb = (np.stack([img] * 3, 0) - np.array(MEAN, np.float32)[:, None, None]) / np.array(STD, np.float32)[:, None, None]
        return torch.from_numpy(rgb.astype(np.float32)), torch.from_numpy(mask)

seg_train = DataLoader(SliceSet(*arrays["train"], True), batch_size=SEG_BATCH, shuffle=True, num_workers=workers, pin_memory=AMP)
seg_val = DataLoader(SliceSet(*arrays["val"], False), batch_size=SEG_BATCH * 2, num_workers=workers, pin_memory=AMP)

## 2 · Train the U-Net

In [ ]:
REGIONS = {"whole_tumor": (1, 2, 3), "tumor_core": (1, 3), "enhancing_tumor": (3,)}

def dice_loss(logits, target, eps=1e-6):
    probs = F.softmax(logits.float(), 1)
    onehot = F.one_hot(target, len(SEG_CLASSES)).permute(0, 3, 1, 2).float()
    inter = (probs * onehot).sum((0, 2, 3))
    denom = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps))[1:].mean()

def run_seg_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    ce = nn.CrossEntropyLoss()
    total = seen = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                logits = model(x)
                loss = ce(logits, y) + dice_loss(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            total += loss.item() * len(x); seen += len(x)
    return total / seen

unet = UNetVGG16().to(DEVICE)
encoder = [p for n_, p in unet.named_parameters() if n_.split(".")[0] in ("enc1", "enc2", "enc3", "enc4", "bottleneck")]
decoder = [p for n_, p in unet.named_parameters() if n_.split(".")[0] not in ("enc1", "enc2", "enc3", "enc4", "bottleneck")]
opt = torch.optim.AdamW([{"params": encoder, "lr": 2e-5}, {"params": decoder, "lr": 3e-4}], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=UNET_EPOCHS)
scaler = torch.amp.GradScaler(enabled=AMP)

best_loss, seg_path = float("inf"), OUT / "unet_vgg16.pt"
resume_path = OUT / "unet_resume.pt"
start_epoch = 1
if resume_path.exists():
    state = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    unet.load_state_dict(state["model"]); opt.load_state_dict(state["opt"])
    sched.load_state_dict(state["sched"]); scaler.load_state_dict(state["scaler"])
    best_loss, start_epoch = state["best_loss"], state["epoch"] + 1
    print(f"resuming after epoch {state['epoch']} (best val {best_loss:.4f})")

for epoch in range(start_epoch, UNET_EPOCHS + 1):
    t = time.time()
    tr = run_seg_epoch(unet, seg_train, opt, scaler)
    va = run_seg_epoch(unet, seg_val)
    sched.step()
    flag = ""
    if va < best_loss:
        best_loss = va
        torch.save({k: v.detach().cpu() for k, v in unet.state_dict().items()}, seg_path)
        flag = "  ← saved"
    torch.save({"epoch": epoch, "model": unet.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "best_loss": best_loss}, resume_path)
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}  {time.time() - t:.0f}s{flag}")

In [ ]:
# Dice per BraTS region on held-out patients, with the best checkpoint.
unet.load_state_dict(torch.load(seg_path, map_location=DEVICE, weights_only=True))
unet.eval()
inter = {k: 0 for k in REGIONS}; denom = {k: 0 for k in REGIONS}
with torch.inference_mode():
    for x, y in seg_val:
        pred = unet(x.to(DEVICE)).argmax(1).cpu()
        for k, labels in REGIONS.items():
            lab = torch.tensor(labels)
            p, g = torch.isin(pred, lab), torch.isin(y, lab)
            inter[k] += (p & g).sum().item(); denom[k] += p.sum().item() + g.sum().item()
seg_metrics = {
    "dice": {k: round(2 * inter[k] / denom[k], 4) if denom[k] else None for k in REGIONS},
    "best_val_loss": round(best_loss, 4),
    "epochs": UNET_EPOCHS,
    "freeze_encoder": False,
    "train_slices": len(seg_train.dataset),
    "val_slices": len(seg_val.dataset),
}
(OUT / "unet_vgg16.metrics.json").write_text(json.dumps(seg_metrics, indent=2))
print("DICE", seg_metrics["dice"])

## 3 · Check and download

In [ ]:
import shutil

# Reload the file the way the backend does (strict key match, weights_only).
u = UNetVGG16(pretrained_encoder=False)
u.load_state_dict(torch.load(OUT / "unet_vgg16.pt", map_location="cpu", weights_only=True))
with torch.inference_mode():
    assert u.eval()(torch.zeros(1, 3, 256, 256)).shape == (1, 4, 256, 256)

pack = WORK / "neuroscan_unet"
pack.mkdir(exist_ok=True)
for name in ("unet_vgg16.pt", "unet_vgg16.metrics.json"):
    shutil.copy2(OUT / name, pack / name)
archive = shutil.make_archive(str(OUT.parent / "neuroscan_unet"), "zip", pack)
for f in sorted(pack.iterdir()):
    print(f"{f.name:<28} {f.stat().st_size / 1e6:7.1f} MB")
print("archive:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)
elif IN_KAGGLE:
    print("Download neuroscan_unet.zip from the Output panel (/kaggle/working).")

## Install on your computer

Put `unet_vgg16.pt` and `unet_vgg16.metrics.json` into `backend/weights/` (next to `vgg16_classifier.pt`):

```bash
cd "Longitutdinal Brain Tumour Segmentation/backend"
unzip -o ~/Downloads/neuroscan_unet.zip -d weights/
```

Then restart the backend.